## **RAG의 핵심, 문서 검색기 Retriever**

### **사용자의 쿼리를 재해석하여 검색하다, MultiQueryRetriever**

**Chroma DB에 문서 벡터 저장**

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
#헌법 PDF 파일 로드
loader = PyPDFLoader(r"대한민국 헌법.pdf")
pages = loader.load_and_split()

#PDF 파일을 500자 청크로 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(pages)

#ChromaDB에 청크들을 벡터 임베딩으로 저장(OpenAI 임베딩 모델 활용)
db = Chroma.from_documents(docs, OpenAIEmbeddings(model = 'text-embedding-3-small'))

**질문을 여러 버전으로 재해석하여 Retriever에 활용**

In [6]:
#```Chroma DB에 대한민국 헌법 PDF 임베딩 변환 및 저장하는 과정은 위 셀에 있습니다```
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

#질문 문장 question으로 저장
question = "국회의원의 의무는 무엇이 있나요?"
#여러 버전의 질문으로 변환하는 역할을 맡을 LLM 선언
llm = ChatOpenAI(model_name="gpt-3.5-turbo-0125",
                 temperature = 0)
#MultiQueryRetriever에 벡터DB 기반 Retriever와 LLM 선언
retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(), llm=llm
)

# 여러 버전의 문장 생성 결과를 확인하기 위한 로깅 과정
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

#여러 버전 질문 생성 결과와 유사 청크 검색 개수 출력
unique_docs = retriever_from_llm.invoke(input=question)
len(unique_docs)

5

In [7]:
unique_docs

[Document(metadata={'moddate': '2017-08-06T23:09:57+09:00', 'producer': 'iText 2.1.7 by 1T3XT', 'source': '대한민국 헌법.pdf', 'creationdate': '2017-08-06T23:09:57+09:00', 'total_pages': 21, 'page_label': '8', 'creator': 'PyPDF', 'page': 7}, page_content='제43조 국회의원은 법률이 정하는 직을 겸할 수 없다.\n \n제44조 ①국회의원은 현행범인인 경우를 제외하고는 회기중 국회의 동의없이 체포 또는 구금되\n지 아니한다.\n②국회의원이  회기전에  체포  또는  구금된  때에는  현행범인이  아닌  한  국회의  요구가  있으면\n회기중  석방된다.\n \n제45조 국회의원은 국회에서 직무상 행한 발언과 표결에 관하여 국회외에서 책임을 지지 아니한\n다.\n \n제46조 ①국회의원은 청렴의 의무가 있다.\n②국회의원은 국가이익을 우선하여 양심에 따라 직무를 행한다.\n③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하\n여 재산상의 권리ㆍ이익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.\n \n제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의 임시회는 대\n통령 또는 국회재적의원 4분의 1 이상의 요구에 의하여 집회된다.'),
 Document(metadata={'total_pages': 21, 'creator': 'PyPDF', 'source': '대한민국 헌법.pdf', 'creationdate': '2017-08-06T23:09:57+09:00', 'page': 9, 'producer': 'iText 2.1.7 by 1T3XT', 'moddate': '2017-08-06T23:09:57+09:00', 'page_label': '10'}, p

### **컨텍스트 재정렬, Long-Context Reorder**

**[Long-Context Reorder 없이 유사 문서 출력]**

In [ ]:
#Chroma dimension 관련 에러 발생 시 실행
# Chroma().delete_collection()

In [13]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 7.7 MB/s eta 0:00:00m eta 0:00:010:01:01m

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_community.document_transformers import (
    LongContextReorder,
)
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI

texts = [
    "바스켓볼은 훌륭한 스포츠입니다.",
    "플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.",
    "셀틱스는 제가 가장 좋아하는 팀입니다.",
    "이것은 보스턴 셀틱스에 관한 문서입니다."
    "저는 단순히 영화 보러 가는 것을 좋아합니다",
    "보스턴 셀틱스가 20점차로 이겼어요",
    "이것은 그냥 임의의 텍스트입니다.",
    "엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.",
    "L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.",
    "래리 버드는 상징적인 NBA 선수였습니다.",
]
# Chroma Retriever 선언(10개의 유사 문서 출력)
retriever = FAISS.from_texts(texts, OpenAIEmbeddings(model = 'text-embedding-3-small')).as_retriever(
    search_kwargs={"k": 10}
)
query = "셀틱에 대해 설명해줘"

# 유사도 기준으로 검색 결과 출력
docs = retriever.invoke(query)
for i in docs:
  print(i.page_content)

셀틱스는 제가 가장 좋아하는 팀입니다.
보스턴 셀틱스가 20점차로 이겼어요
이것은 보스턴 셀틱스에 관한 문서입니다.저는 단순히 영화 보러 가는 것을 좋아합니다
L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.
이것은 그냥 임의의 텍스트입니다.
바스켓볼은 훌륭한 스포츠입니다.
엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.
플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.
래리 버드는 상징적인 NBA 선수였습니다.


**[Long-Context Reorder 활용하여 유사 문서 출력]**

In [15]:
#LongContextReorder 선언
reordering = LongContextReorder()
#검색된 유사문서 중 관련도가 높은 문서를 맨앞과 맨뒤에 재정배치
reordered_docs = reordering.transform_documents(docs)
for i in reordered_docs:
  print(i.page_content)

셀틱스는 제가 가장 좋아하는 팀입니다.
이것은 보스턴 셀틱스에 관한 문서입니다.저는 단순히 영화 보러 가는 것을 좋아합니다
이것은 그냥 임의의 텍스트입니다.
엘든 링은 지난 15 년 동안 최고의 게임 중 하나입니다.
래리 버드는 상징적인 NBA 선수였습니다.
플라이 미 투 더 문은 제가 가장 좋아하는 노래 중 하나입니다.
바스켓볼은 훌륭한 스포츠입니다.
L. 코넷은 최고의 셀틱스 선수 중 한 명입니다.
보스턴 셀틱스가 20점차로 이겼어요
